In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [8]:
!pip install imbalanced-learn==0.11.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.6/235.6 kB 5.1 MB/s eta 0:00:0000:01


In [3]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.expand_frame_repr', False)

In [12]:
!pip install mlflow dagshub --quiet
import mlflow
from dagshub import dagshub_logger
import os

# Set tracking URI manually
mlflow.set_tracking_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

# Use your DagsHub credentials
os.environ["MLFLOW_TRACKING_USERNAME"] = "ekvirika"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "3f601f2c2c7a6bca448ebc69f4f5d4b49daffd8f"

# Optional: set registry if you're using model registry
mlflow.set_registry_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

In [13]:
import mlflow
mlflow.set_experiment("LogReg_Training")

<Experiment: artifact_location='mlflow-artifacts:/b1d51fd536524d16a2033d9de3976de0', creation_time=1744552935348, experiment_id='0', last_update_time=1744552935348, lifecycle_stage='active', name='LogReg_Training', tags={}>

In [17]:
df = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")
df_id = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")
df = df.merge(df_id, how='left', on='TransactionID')

# Cleaning

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

In [6]:

class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.columns_to_drop_ = []

    def fit(self, X, y=None):
        # Convert to DataFrame if it's a NumPy array
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        corr_matrix = X.corr().abs()
        upper = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        upper_tri = corr_matrix.where(upper)

        self.columns_to_drop_ = [column for column in upper_tri.columns if any(upper_tri[column] > self.threshold)]
        return self

    def transform(self, X):
        # Convert to DataFrame if it's a NumPy array
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        return X.drop(columns=self.columns_to_drop_, errors='ignore')


In [28]:
class DropHighNulls(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        # If X is a pandas DataFrame, use .columns, else use column indices
        if isinstance(X, pd.DataFrame):
            self.to_drop_ = X.columns[X.isnull().mean() > self.threshold].tolist()
        else:
            self.to_drop_ = [i for i in range(X.shape[1]) if np.isnan(X[:, i]).mean() > self.threshold]
        return self
    
    def transform(self, X):
        # Drop columns based on the computed 'to_drop_' list
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.to_drop_)
        else:
            return np.delete(X, self.to_drop_, axis=1)


# Feature Engineering

In [10]:
class UserIDCreator(BaseEstimator, TransformerMixin):
    def __init__(self, version=1):
        self.version = version

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.version == 1:
            X['user_id'] = X['card1'].astype(str) + '_' + \
                           X['Transaction_day'].astype(str) + '_' + \
                           X['Transaction_hour'].astype(str)
        else:
            X['user_id'] = X['card1'].astype(str) + '_' + \
                           X['P_emaildomain'].astype(str) + '_' + \
                           X['Transaction_day'].astype(str)
        return X


In [24]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class TransactionDateTransformer(BaseEstimator, TransformerMixin):
    """
    Extracts day, week, month, and hour features from TransactionDT in seconds using math only.
    No datetime module is used.
    """
    def __init__(self, seconds_per_day=86400):
        self.seconds_per_day = seconds_per_day

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        seconds = X['TransactionDT'].astype(np.int64)
        
        # Days since start (e.g. start = day 0)
        X['Transaction_day'] = (seconds // self.seconds_per_day).astype(np.int16)
        X['Transaction_week'] = (seconds // (self.seconds_per_day * 7)).astype(np.int16)
        X['Transaction_month'] = (seconds // (self.seconds_per_day * 30)).astype(np.int16)
        X['Transaction_hour'] = ((seconds % self.seconds_per_day) // 3600).astype(np.int8)

        return X

## pipeline

In [18]:
target = "isFraud"
X = df.drop(columns=[target])
y = df[target]

In [19]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

In [20]:
# Pipeline for numerical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Pipeline for categorical features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse=False))
])

# Combine them
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

In [26]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

# Full pipeline
full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("drop_nulls", DropHighNulls(threshold=0.9)),
    ('date', TransactionDateTransformer()),
    ('user_id', UserIDCreator(version=1)),  # version 1 for user_id
    ("feature_selector", SelectKBest(score_func=f_classif, k=50)),  # You can tune k
    ("classifier", LogisticRegression(max_iter=1000))
])

# Training + Feature Selection using SelectKBest

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

mlflow.start_run(run_name="LogReg_Training")
X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [ ]:
full_pipeline.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [ ]:
y_pred = full_pipeline.predict(X_valid)
print(classification_report(y_valid, y_pred))

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import WOEEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from sklearn.feature_selection import SelectKBest, f_classif
import mlflow
import mlflow.sklearn
import joblib
from sklearn.feature_selection import RFE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform


# ---------------------- Load Data ----------------------
with mlflow.start_run(run_name="LogReg_Cleaning"):

    df = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")
    df_identity = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")
    df = df.merge(df_identity, how='left', on='TransactionID')
    
    # run_id = '78bbe86508204c1388aaa3ae689133df'
    # artifact_path = "user_id_datasets/card1_addr1/dataset_card1_addr1.parquet"
    # local_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_path)

    # df = pd.read_parquet(local_path)
    mlflow.log_param("initial_columns", df.shape[1])

    # Target column
    target = "isFraud"
    y = df[target]
    X = df.drop(columns=[target])

    # ---------------------- Drop High-Null Columns ----------------------
    missing_ratio = X.isnull().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.95].index.tolist()
    
    X.drop(columns=cols_to_drop, inplace=True)
    mlflow.log_param("dropped_columns", len(cols_to_drop))
    mlflow.log_param("remaining_columns_after_drop", X.shape[1])

    # ---------------------- Identify Feature Types ----------------------
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    mlflow.log_param("num_categorical_features", len(cat_cols))
    mlflow.log_param("num_numerical_features", len(num_cols))

    # ---------------------- Train/Test Split ----------------------
    user_list = df["user_id"].unique()
    train_users, valid_users = train_test_split(user_list, test_size=0.2, random_state=42)

    train_mask = df["user_id"].isin(train_users)
    valid_mask = df["user_id"].isin(valid_users)

    X_train = df[train_mask].drop(columns=["isFraud"])
    y_train = df[train_mask]["isFraud"]

    X_valid = df[valid_mask].drop(columns=["isFraud"])
    y_valid = df[valid_mask]["isFraud"]


    # ---------------------- Preprocessing ----------------------
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        # ("encoder", OneHotEncoder(handle_unknown="ignore", sparse=False))
        ("encoder", WOEEncoder())
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])

    corr_filter = CorrelationFilter(threshold=0.95)
    
    # ---------------------- Full Pipeline ----------------------

    # rfe_model = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=50)

    # model_pipeline = Pipeline([
    #     ("preprocessing", preprocessor),
    #     ("corr_filter", corr_filter),
    #     # ("oversample", RandomOverSampler(random_state=42)),
    #     ("feature_selection", rfe_model),  # Tuneable
    #     ("classifier", LogisticRegression(max_iter=1000))
    # ])

    model = LogisticRegression(
        C=0.1,  # Regularization strength
        penalty='l2',  # Try 'elasticnet' with l1_ratio too
        solver='saga',
        class_weight='balanced',
        max_iter=2500
    )

    model_pipeline = ImbPipeline([
        ("preprocessing", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=20)),
        # Add upsampling step - using RandomOverSampler to achieve ~60% negatives
        # ("upsampling", RandomOverSampler( random_state=42)),
        # Alternatively, you can use SMOTE which creates synthetic samples
        ("upsampling", SMOTE(sampling_strategy=0.67, random_state=42)),
        ("classifier", model)
    ])


    param_grid = {
        "classifier__C": loguniform(1e-3, 1e2),
        "classifier__penalty": ["l2"],  # or 'elasticnet' if you add l1_ratio
        "classifier__solver": ["saga"],
        "feature_selection__k": [30, 40],
    }

    search = RandomizedSearchCV(
        estimator=model_pipeline,
        param_distributions=param_grid,
        n_iter=20,
        cv=3,
        scoring='roc_auc',
        verbose=1,
        n_jobs=-1,
        random_state=42,
    )


    # ---------------------- Train ----------------------
    # model_pipeline.fit(X_train, y_train)
    search.fit(X_train, y_train)
    # preprocessed_feature_names = model_pipeline.named_steps["preprocessing"].get_feature_names_out()
    # rfe_mask = model_pipeline.named_steps["feature_selection"].support_
    # selected_features = preprocessed_feature_names[rfe_mask]


    # ---------------------- Evaluation ----------------------
    y_pred = model_pipeline.predict(X_valid)
    y_proba = model_pipeline.predict_proba(X_valid)[:, 1]

    auc = roc_auc_score(y_valid, y_proba)
    f1 = f1_score(y_valid, y_pred)

    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("f1_score", f1)

    report = classification_report(y_valid, y_pred, output_dict=True)
    for cls, metrics in report.items():
        if isinstance(metrics, dict):
            for m_name, val in metrics.items():
                mlflow.log_metric(f"{cls}_{m_name}", val)

    # ---------------------- Log the model ----------------------

    # mlflow.log_param("final_feature_count", len(selected_features))
    # mlflow.log_param("selected_features", ",".join(selected_features))

    mlflow.sklearn.log_model(model_pipeline, "logistic_model_pipeline")

KeyError: 'user_id'

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

# Custom transformer to extract user-related time-based features
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class UserIDTimeExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Check if X is a DataFrame or a NumPy array
        if isinstance(X, pd.DataFrame):
            # X is a DataFrame, so we can directly check for column names
            if 'user_id' in X.columns:
                if 'TransactionDT' in X.columns:
                    X['TransactionDT'] = pd.to_datetime(X['TransactionDT'], unit='s')
                    X['hour_of_day'] = X['TransactionDT'].dt.hour
                    X['day_of_week'] = X['TransactionDT'].dt.dayofweek
                    X['is_weekend'] = (X['day_of_week'] >= 5).astype(int)
                X['user_id_count'] = X.groupby('user_id')['user_id'].transform('count')
            return X
        elif isinstance(X, np.ndarray):
            # X is a NumPy array, so we can't check column names directly
            # We can only access the data, so skip the user_id-based feature extraction
            return X
        else:
            raise TypeError("Input should be a Pandas DataFrame or a NumPy ndarray.")


# ---------------------- Full Pipeline ----------------------

# Define pre-processing pipelines for numerical and categorical features
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", WOEEncoder())
])

# Define full preprocessor
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
])

# Add UserID Time Extractor in the pipeline
model_pipeline = ImbPipeline([
    ("preprocessing", preprocessor),
    ("user_id_time_extractor", UserIDTimeExtractor()),  # Add our custom transformer
    ("feature_selection", SelectKBest(score_func=f_classif, k=20)),
    ("upsampling", SMOTE(sampling_strategy=0.67, random_state=42)),
    ("classifier", LogisticRegression(C=0.1, penalty='l2', solver='saga', class_weight='balanced', max_iter=2500))
])

# Train the model
model_pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['TransactionID',
                                                   'TransactionDT',
                                                   'TransactionAmt', 'card1',
                                                   'card2', 'card3', 'card5',
                                                   'addr1', 'addr2', 'dist1',
                                                   'dist2', 'C1', 'C2', 'C3',
                                                   'C4', 'C5', 'C6', 'C7', 'C8',
                                                   'C9', 'C10', 'C11', 'C12',
                                                   'C13', 'C14...
                                                   'id_16', 'id_28', 'id_29',
                                                   'id_30', 'id_31', 'id_33',
                                                   'id_34', 'id_35', 'id_36',
                                                   'id_37', 'id_38',
                                                   'DeviceType',
                                                   'DeviceInfo'])])),
                ('user_id_time_extractor', UserIDTimeExtractor()),
                ('feature_selection', SelectKBest(k=20)),
                ('upsampling', SMOTE(random_state=42, sampling_strategy=0.67)),
                ('classifier',
                 LogisticRegression(C=0.1, class_weight='balanced',
                                    max_iter=2500, solver='saga'))])

In [9]:
mlflow.set_experiment("LogReg_Training")

<Experiment: artifact_location='file:///kaggle/working/mlruns/957850388063184577', creation_time=1745967886671, experiment_id='957850388063184577', last_update_time=1745967886671, lifecycle_stage='active', name='LogReg_Training', tags={}>

In [17]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, f1_score, classification_report

# ---------------------- Evaluation ----------------------
y_pred = model_pipeline.predict(X_valid)
y_proba = model_pipeline.predict_proba(X_valid)[:, 1]

# Calculate metrics
auc = roc_auc_score(y_valid, y_proba)
f1 = f1_score(y_valid, y_pred)

# Start MLflow run with custom name
with mlflow.start_run(run_name="final_run"):
    # Log metrics
    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("f1_score", f1)

    # Log classification report as metrics
    report = classification_report(y_valid, y_pred, output_dict=True)
    for cls, metrics in report.items():
        if isinstance(metrics, dict):
            for m_name, val in metrics.items():
                mlflow.log_metric(f"{cls}_{m_name}", val)

    # ---------------------- Log and Register the model ----------------------
    # Log the model pipeline to MLflow
    mlflow.sklearn.log_model(model_pipeline, "logistic_model_pipeline")
    
    # Get the URI of the logged model
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/logistic_model_pipeline"
    
    # Register the model to the Model Registry
    model_name = "fraud_detection_model"  # You can change this name
    model_registered = mlflow.register_model(model_uri, model_name)

    print(f"Model registered with name: {model_registered.name}, version: {model_registered.version}")


2025/04/29 23:16:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'fraud_detection_model'.
2025/04/29 23:16:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud_detection_model, version 1
Created version '1' of model 'fraud_detection_model'.


Model registered with name: fraud_detection_model, version: 1
🏃 View run final_run at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/0/runs/232cd843553f40ff81ce556f6e3e4b78
🧪 View experiment at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/0
